<a href="https://colab.research.google.com/github/yashsinghal1234/twitter_sentiment_analysis/blob/main/twitter_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import nltk
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from wordcloud import WordCloud

nltk.download('stopwords')

train = pd.read_csv('train_tweet.csv')
test = pd.read_csv('test_tweets.csv')


print(train.isnull().sum())
print(test.isnull().sum())

# Data Preprocessing
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stopwords.words('english')])
    return text

train['clean_tweet'] = train['tweet'].apply(clean_tweet)
test['clean_tweet'] = test['tweet'].apply(clean_tweet)


vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))  # Unigrams and bigrams
X = vectorizer.fit_transform(train['clean_tweet']).toarray()
y = train['label']


X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)

# Model Selection and Hyperparameter Tuning
models = {
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree': DecisionTreeClassifier(),
    'SVC': SVC(),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    print(f"{model_name} - Training Accuracy: {model.score(X_train, y_train):.4f}")
    print(f"{model_name} - Validation Accuracy: {model.score(X_valid, y_valid):.4f}")
    print(f"{model_name} - F1 Score: {f1_score(y_valid, y_pred, average='weighted'):.4f}")
    print(f"{model_name} - Confusion Matrix:\n{confusion_matrix(y_valid, y_pred)}\n")


def manual_testing(tweet):
    cleaned_tweet = clean_tweet(tweet)
    tweet_vector = vectorizer.transform([cleaned_tweet])
    tweet_vector = scaler.transform(tweet_vector)
    prediction = model.predict(tweet_vector)
    return prediction


test_tweet = input("Enter a tweet for sentiment prediction:\n")
print("Predicted Sentiment:", manual_testing(test_tweet))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


id       0
label    0
tweet    0
dtype: int64
id       0
tweet    0
dtype: int64
Random Forest - Training Accuracy: 0.9990
Random Forest - Validation Accuracy: 0.9558
Random Forest - F1 Score: 0.9496
Random Forest - Confusion Matrix:
[[7384   48]
 [ 305  254]]

Logistic Regression - Training Accuracy: 0.9974
Logistic Regression - Validation Accuracy: 0.9263
Logistic Regression - F1 Score: 0.9302
Logistic Regression - Confusion Matrix:
[[7065  367]
 [ 222  337]]

Decision Tree - Training Accuracy: 0.9992
Decision Tree - Validation Accuracy: 0.9409
Decision Tree - F1 Score: 0.9397
Decision Tree - Confusion Matrix:
[[7220  212]
 [ 260  299]]

SVC - Training Accuracy: 0.9776
SVC - Validation Accuracy: 0.9531
SVC - F1 Score: 0.9433
SVC - Confusion Matrix:
[[7415   17]
 [ 358  201]]



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [18:27:48] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost - Training Accuracy: 0.9619
XGBoost - Validation Accuracy: 0.9514
XGBoost - F1 Score: 0.9429
XGBoost - Confusion Matrix:
[[7389   43]
 [ 345  214]]

Enter a tweet for sentiment prediction:
i love pizza of italy
Predicted Sentiment: [1]
